<a href="https://colab.research.google.com/github/nnott3/KilterTransformer/blob/main/gpt_shuffle_me.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
!git clone https://github.com/nnott3/KilterTransformer.git
%cd KilterTransformer
!ls

/content/KilterTransformer
bert_improved.ipynb  gpt.ipynb		   project_structure  utils_old
bert.ipynb	     gpt_shuffle.ipynb	   pyproject.toml     uv.lock
data		     gpt_shuffle_me.ipynb  readme.md	      wandb
EDA.ipynb	     gpt_wandb.ipynb	   req
figs		     main.ipynb		   saved_models
gitignore	     models		   src


# init

In [6]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from datetime import datetime
import random
from functools import partial
from typing import List, Dict

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import (
    GPT2LMHeadModel,
    GPT2Config,
    PreTrainedTokenizerFast,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)
import numpy as np
from datasets import Dataset, disable_progress_bar
import wandb
from transformers import DataCollatorForLanguageModeling
from src.data_processing import DataPreprocessing
from src.tokenizer import train_tokenizer
from src.gpt import preprocess_datasets, KilterGPT
import warnings

warnings.filterwarnings("ignore",)
disable_progress_bar()



In [7]:
wandb.init(
    project="climb-gpt-shuffle",
    name=f"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}",
    config={
        "architecture": "GPT-SetLoss",
        "n_embd": 256,
        "n_head": 4,
        "n_layer": 6,
        "n_positions": 128,
        "dropout": 0.1,
        "epochs": 30,
        "batch_size": 16,
        "learning_rate": 1e-4,
        "weight_decay": 0.01,
        "gradient_accumulation_steps": 1,
        "early_stopping_patience": 5,
        "allow_empty_prompt": True,
        "min_prefix_len": 1,  # Changed from 3 to allow BOS-only
    }
)

run_name = wandb.run.name

OUT_DIR = f"/content/drive/MyDrive/KilterTransformer/models/climb_gpt/{run_name}"
# OUT_DIR = f"models/climb_gpt/{run_name}"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

wandb: Currently logged in as: treepatchantaurai (treepatchantaurai-me) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Using device: cuda


# data

In [8]:
# Load and split data
dp = DataPreprocessing()
datasets = dp.load_climbs()

# 80-10-10 split
train_test = datasets.train_test_split(test_size=0.2, seed=42)
val_test = train_test['test'].train_test_split(test_size=0.5, seed=42)

datasets = {
    'train': train_test['train'],
    'val': val_test['train'],
    'test': val_test['test']
}

print(f"Train size: {len(datasets['train'])}")
print(f"Val size: {len(datasets['val'])}")
print(f"Test size: {len(datasets['test'])}")

Loaded 76992 routes from cache data/climbs_cleaned.csv
Train size: 61593
Val size: 7699
Test size: 7700


In [9]:
# Train tokenizer
tokenizer = train_tokenizer(datasets, OUT_DIR)
wandb.config.update({"vocab_size": tokenizer.vocab_size})

print(f"Vocabulary size: {tokenizer.vocab_size}")
print(f"Special tokens: {tokenizer.special_tokens_map}")

# Tokenize datasets
datasets = preprocess_datasets(datasets, tokenizer) # Dataset object
print("✓ Datasets tokenized")

Built vocabulary with 1932 tokens (1928 holds)

Vocab size: 1932 tokens
First 10 tokens: [('hand1542', 1702), ('start1138', 289), ('hand1296', 922), ('feet1332', 1064), ('feet1283', 868), ('start1552', 1741), ('feet1587', 1880), ('grade27', 27), ('feet1373', 1228), ('start1100', 137)]

Sample encodings:

Input: angle35_grade14_feet1595_start1400
Tokens: ['[BOS]', 'angle35', 'grade14', 'feet1595', '[UNK]', '[EOS]', ('[PAD]', 19)]

Input: angle40_grade15_feet1595_start1596_hand1597_finish1598
Tokens: ['[BOS]', 'angle40', 'grade15', 'feet1595', 'start1596', 'hand1597', 'finish1598', '[EOS]', ('[PAD]', 17)]
Saving tokenizer to /content/drive/MyDrive/KilterTransformer/models/climb_gpt/run_20251104_185401
Vocabulary size: 1932
Special tokens: {'bos_token': '[BOS]', 'eos_token': '[EOS]', 'unk_token': '[UNK]', 'pad_token': '[PAD]'}
✓ Datasets tokenized


In [10]:
example = datasets["train"][0]
for k, v in example.items():
    print(f"{k:15s} -> {v[:10]} ... len={len(v)}")


input_ids       -> [1, 9, 23, 56, 64, 333, 674, 862, 1070, 1122] ... len=13
token_type_ids  -> [0, 0, 0, 0, 0, 0, 0, 0, 0, 0] ... len=13
attention_mask  -> [1, 1, 1, 1, 1, 1, 1, 1, 1, 1] ... len=13


# gpt -default

In [11]:
class KilterGPT(nn.Module):
    """GPT-2 model for generating Kilter Board climbing routes."""
    def __init__(
        self,
        vocab_size: int,
        n_embd: int = 192,
        n_head: int = 3,
        n_layer: int = 3,
        n_positions: int = 128,
        dropout: float = 0.1
    ):
        super().__init__()
        config = GPT2Config(
            vocab_size=vocab_size,
            n_embd=n_embd,
            n_head=n_head,
            n_layer=n_layer,
            n_positions=n_positions,
            n_ctx=n_positions,
            resid_pdrop=dropout,
            embd_pdrop=dropout,
            attn_pdrop=dropout,
        )
        self.model = GPT2LMHeadModel(config)
        self.config = config

    def forward(self, input_ids, attention_mask=None, labels=None, token_type_ids=None):
        # GPT2LMHeadModel already handles label shifting internally
        # when you pass labels=labels. So don’t manually shift them here.
        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )

        return outputs


In [12]:
model = KilterGPT(
    vocab_size=tokenizer.vocab_size,
    n_embd=256,
    n_head=4,
    n_layer=6,
    n_positions=128,
    dropout=0.1
    )
training_args = TrainingArguments(
    output_dir=OUT_DIR,
    eval_strategy="steps",
    save_strategy="steps",
    save_total_limit=3,
    overwrite_output_dir=True,
    logging_steps=100,  # Log more frequently for wandb
    eval_steps=3000,
    save_steps=3000,
    num_train_epochs=10,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=1,
    learning_rate=1e-5,
    weight_decay=0.01,
    adam_beta1=0.9,
    adam_beta2=0.999,
    report_to="wandb",  # Enable wandb reporting
    remove_unused_columns=False,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_dir=f"{OUT_DIR}/logs",
    load_best_model_at_end=True,
    dataloader_pin_memory=False,
    save_safetensors=False,
    run_name=run_name,
    )
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=datasets["train"],
    eval_dataset=datasets["val"],
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)],
    )

In [13]:
trainer.train()

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
3000,4.589300,4.548358
6000,4.157400,4.112203
9000,3.934400,3.911718
12000,3.830800,3.796501
15000,3.732100,3.711902
18000,3.699100,3.656183
21000,3.650700,3.612122
24000,3.625500,3.581848
27000,3.585000,3.557393
30000,3.566200,3.540096


TrainOutput(global_step=38500, training_loss=3.8686451503208708, metrics={'train_runtime': 1165.5778, 'train_samples_per_second': 528.433, 'train_steps_per_second': 33.031, 'total_flos': 0.0, 'train_loss': 3.8686451503208708, 'epoch': 10.0})

# test

In [16]:
def test_loss_gen(trainer, model):

  pred = trainer.predict(datasets["test"])

  # Unpack
  logits = torch.tensor(pred.predictions)  # (B, L, V)
  labels = torch.tensor(pred.label_ids)    # (B, L)
  B, L, V = logits.shape

  # ------------------------------------------------------
  # Compute standard LM loss (same as GPT2)
  # shift so that tokens < t predict token t
  shift_logits = logits[..., :-1, :].contiguous()
  shift_labels = labels[..., 1:].contiguous()

  # ignore padding
  loss_fct = torch.nn.CrossEntropyLoss(ignore_index=-100)
  loss = loss_fct(shift_logits.view(-1, V), shift_labels.view(-1))
  print(f"Manual test loss: {loss.item():.4f}")
  # ------------------------------------------------------


  device = "cuda" if torch.cuda.is_available() else "cpu"
  model = model.model #use underlying GPT2LMHeadModel instead of KilterGPT (unless implement .generate() later)
  model.to(device)
  model.eval()

  test_prompts = [
      "[BOS]",
      "[BOS] angle40",
      "[BOS] angle40 grade15",
      "[BOS] angle40 grade15 start1139",
      "[BOS] angle40 grade15 hand1315",
  ]

  # generation config
  gen_kwargs = dict(
      max_new_tokens=20,
      do_sample=False,        # deterministic for inspection
      temperature=1.0,
      top_p=0.9,
      pad_token_id=tokenizer.eos_token_id,
      eos_token_id=tokenizer.convert_tokens_to_ids("[EOS]"),
  )

  for i, prompt in enumerate(test_prompts):
      input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
      with torch.no_grad():
          output = model.generate(input_ids, **gen_kwargs)

      decoded = tokenizer.decode(output[0], skip_special_tokens=False)
      print(f"\n--- Example {i} ---")
      print(f"Context: {prompt}")
      print(f"Generated: {decoded}")

      # Optional: visualize token-by-token
      tokens = tokenizer.convert_ids_to_tokens(output[0])
      print("Tokens:", tokens)


In [15]:
test_loss_gen(trainer, model)

The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Manual test loss: 3.5244

--- Example 0 ---
Context: [BOS]
Generated: [BOS] [BOS] [EOS] [EOS]
Tokens: ['[BOS]', '[BOS]', '[EOS]', '[EOS]']

--- Example 1 ---
Context: [BOS] angle40
Generated: [BOS] [BOS] angle40 [EOS] [EOS]
Tokens: ['[BOS]', '[BOS]', 'angle40', '[EOS]', '[EOS]']

--- Example 2 ---
Context: [BOS] angle40 grade15
Generated: [BOS] [BOS] angle40 grade15 [EOS] [EOS]
Tokens: ['[BOS]', '[BOS]', 'angle40', 'grade15', '[EOS]', '[EOS]']

--- Example 3 ---
Context: [BOS] angle40 grade15 start1139
Generated: [BOS] [BOS] angle40 grade15 start1139 [EOS] [EOS]
Tokens: ['[BOS]', '[BOS]', 'angle40', 'grade15', 'start1139', '[EOS]', '[EOS]']

--- Example 4 ---
Context: [BOS] angle40 grade15 hand1315
Generated: [BOS] [BOS] angle40 grade15 hand1315 [EOS] [EOS]
Tokens: ['[BOS]', '[BOS]', 'angle40', 'grade15', 'hand1315', '[EOS]', '[EOS]']


# on-the-fly augment

In [17]:
class AugmentedRouteDataset(torch.utils.data.Dataset):
    def __init__(self, hf_dataset, tokenizer, num_augments=1):
        self.ds = hf_dataset
        self.tokenizer = tokenizer
        self.num_augments = num_augments
        self.n = len(hf_dataset)

    def __len__(self):
        return self.n * (self.num_augments + 1)  # originals + N augments

    def __getitem__(self, idx):
        base_idx = idx % self.n      # which real example in the base dataset
        aug_idx  = idx // self.n     # which augmentation slot (0 = original)

        example = self.ds[base_idx]
        tokens = example["input_ids"]

        if aug_idx > 0:  # 0 = original, others = augmented
            eos_idx = tokens.index(self.tokenizer.eos_token_id) if self.tokenizer.eos_token_id in tokens else len(tokens)
            prefix = tokens[:3]
            holds = tokens[3:eos_idx]
            tail = tokens[eos_idx:]

            random.shuffle(holds)          # in-place
            tokens = prefix + holds + tail  # use the shuffled list


        max_len = tokenizer.model_max_length  # or a fixed number

        example["input_ids"] = tokens + [tokenizer.pad_token_id] * (max_len - len(tokens))
        example["labels"] = tokens + [-100] * (max_len - len(tokens)) #ignore_index for padding
        example["token_type_ids"] = [0] * max_len
        example["attention_mask"] = [1] * len(tokens) + [0] * (max_len - len(tokens))


        return example

augmented_dataset = AugmentedRouteDataset(datasets["train"], tokenizer, num_augments=1)
trainer.train_dataset = augmented_dataset

In [ ]:
trainer.train()

Step,Training Loss,Validation Loss
3000,4.413600,3.588978
6000,4.393000,3.563963
9000,4.355800,3.539241
12000,4.316700,3.516454
15000,4.289700,3.497188
18000,4.263800,3.482609
21000,4.250200,3.463749
24000,4.221300,3.456135
27000,4.209900,3.444286
30000,4.201100,3.428533


In [ ]:
test_loss_gen(trainer, model)

# order-invariant loss
Instead of predicting the *exact next token* (as in normal language modeling),
we want the model to predict **any valid next hold** — since order among holds doesn't matter.

This means at position *i*, the valid next tokens are **all remaining holds** in the sequence.

Shape explanation:
- logits: `(B, L, V)`  → batch_size × seq_len × vocab_size
- labels: `(B, L)`     → each position contains the target token index or -100 for padding

We’ll compute the log-probability of the **set of valid tokens** instead of a single token.



In [96]:
import torch
import torch.nn.functional as F

# def any_of_next_token_loss(logits, shift_labels, ignore_index=-100, print_debug=False):
#     """
#     logits:   Tensor shape (B, L, V)  -- already shifted logits for predicting next token
#               (e.g. logits[..., :-1, :] from model outputs)
#     shift_labels: Tensor shape (B, L) -- labels shifted left (labels[...,1:])
#                   entries are vocab indices or ignore_index
#     Returns:
#       loss: scalar tensor (mean over valid positions)
#     """
#     B, L, V = logits.shape
#     device = logits.device

#     # Convert logits -> log_probs
#     log_probs = F.log_softmax(logits, dim=-1)  # (B, L, V)

#     # Build valid-mask per position that marks every token that occurs later in the sequence.
#     # For each batch b and position i we want all labels in shift_labels[b, i:] (excluding ignore_index).
#     # We'll build valid_mask by iterating over positions (L is small, so it's fine).
#     valid_mask = torch.zeros((B, L, V), dtype=torch.bool, device=device)

#     for pos in range(L):
#         # tail_labels: shape (B, tail_len)
#         tail_labels = shift_labels[:, pos:]  # (B, L-pos)

#         # flatten to indices with mask of non-ignore
#         flat = tail_labels.reshape(-1)  # (B*(L-pos),)
#         keep = flat != ignore_index
#         if keep.any():
#             kept_indices = flat[keep].long()  # the vocabulary indices we want to mark
#             # To place them back into (B, pos) positions we need which batch each came from:
#             # compute batch index for each row in tail_labels
#             # create tensor of batch ids repeated for each position in tail
#             batch_ids = torch.arange(B, device=device).unsqueeze(1).expand(B, L-pos).reshape(-1)[keep]
#             # set valid_mask[batch_ids, pos, kept_indices] = True
#             valid_mask[batch_ids, pos, kept_indices] = True

#     # Now for each position we may have zero valid tokens (should be ignored)
#     has_valid = valid_mask.any(dim=-1)  # (B, L) bool

#     # For positions with no valid tokens, set masked logits to -inf so logsumexp -> -inf.
#     neg_inf = -1e9
#     masked_log_probs = torch.where(valid_mask, log_probs, neg_inf)  # (B, L, V)

#     # Compute logsumexp over vocab dim -> log probability mass on valid tokens
#     log_sum = torch.logsumexp(masked_log_probs, dim=-1)  # (B, L)

#     # Positions without valid tokens will have log_sum approx = neg_inf. We mask them out.
#     valid_log_sum = log_sum[has_valid]  # 1D tensor of only valid positions

#     if valid_log_sum.numel() == 0:
#         # no valid positions: return zero or raise depending on preference
#         return torch.tensor(0.0, device=device, requires_grad=True)

#     loss = - valid_log_sum.mean()


#     # if print_debug:
#     #     print("=== DEBUG (loop version) ===")
#     #     print("logits.shape:", logits.shape)
#     #     print("shift_labels:\n", shift_labels)
#     #     print("valid_mask.sum per pos:", valid_mask.sum(-1))
#     #     print("has_valid:", has_valid)
#     #     print("log_sum:", log_sum)
#     #     print("loss:", loss.item())
#     return loss

"""
VECTORIZED LOSS: Using Upper-Triangular Masks

Key insight: For each position i, valid targets are ALL tokens at positions j≥i

Example sequence: [BOS, angle, hold1, hold2, hold3, EOS]
Position:           0     1      2      3      4     5

Future mask (upper triangular, each row shows what position i can see):
     j:  0  1  2  3  4  5
   i=0 [[1, 1, 1, 1, 1, 1],  ← pos 0 sees all future
   i=1  [0, 1, 1, 1, 1, 1],  ← pos 1 sees 1-5
   i=2  [0, 0, 1, 1, 1, 1],  ← pos 2 sees 2-5
   i=3  [0, 0, 0, 1, 1, 1],  ← pos 3 sees 3-5
   i=4  [0, 0, 0, 0, 1, 1],  ← pos 4 sees 4-5
   i=5  [0, 0, 0, 0, 0, 1]]  ← pos 5 sees 5 only
"""
def any_of_next_token_loss_vectorized(logits, shift_labels, ignore_index=-100,
                                       bos_token_id=1, eos_token_id=2, print_debug=False):
    """
    Order-invariant loss for climbing routes.

    Loss behavior:
    - BOS (token_id=1): Never predicted (always first in sequence)
    - Angle, Grade, Holds: Set-loss (order-invariant, any remaining token is valid)
    - EOS (token_id=2): Excluded from valid set (standard single-target prediction)
    - Padding: Ignored

    Args:
        logits: (B, L, V) # batch_size, seq_len, vocab_size
        shift_labels: (B, L)
        ignore_index: Padding marker (default -100)
        bos_token_id: BOS token ID (default 1)
        eos_token_id: EOS token ID (default 2)
    """
    B, L, V = logits.shape
    device = logits.device

    # Compute log probabilities
    log_probs = F.log_softmax(logits, dim=-1)  # (B, L, V)

    # mask for valid label positions
    # [TRUE, TRUE, TRUE, TRUE, FALSE, FALSE, ...] where FALSE are PADs
    valid_positions = (shift_labels != ignore_index)  # (B, L)


    # upper-triangle of 1 size (L, L)
    future_mask = torch.triu(torch.ones((L, L), device=device, dtype=torch.bool))

    # Combine: for each sequence, each position sees all future tokens that are valid
    # Expand: valid_positions.unsqueeze(1): (B, L) → (B, 1, L)
    # valid_future_mask[b, i, j] = True if label[j] is valid future of i and j >= i
    valid_future_mask = valid_positions.unsqueeze(1) & future_mask  # (B, L, L)
    # valid_future_mask[0][0] => [TRUE, TRUE, TRUE, ..., TRUE, FALSE, ..., FALSE] (True for all valid tokens + False for PADs)
    # valid_future_mask[0][1] => [FALSE, TRUE, TRUE, ..., TRUE, FALSE, ..., FALSE]
    # valid_future_mask[0][2] => [FALSE, FALSE, TRUE, ..., TRUE, FALSE, ..., FALSE]
    # valid_future_mask[0][-seq_len] => [FALSE, FALSE, FALSE, ..., FALSE, FALSE, ..., FALSE]


    labels_expanded = shift_labels.unsqueeze(1).expand(-1, L, -1)  # (B, L, L)
    # labels_expanded[0] => (B, L) 2d array, row count = batch_size B=16
    # each row labels_expanded[0][0] is [token0, token1, token2, ..., token_seq_len, -100, -100, ...] (len L=21)

    # Set ignored ones to -1
    labels_expanded = torch.where(valid_future_mask, labels_expanded, -torch.ones_like(labels_expanded))
    # each row labels_expanded[0][0] is now [token0, token1, token2, ..., token_seq_len, -1, -1, ...] (len L=21)
    # labels_expanded[0][0] => [token0, token1, token2, ..., token_seq_len, -1, -1, ...] (len L=21)
    # labels_expanded[0][1] => [-1,     token1, token2, ..., token_seq_len, -1, -1, ...]
    # labels_expanded[0][2] => [-1,       -1,   token2, ..., token_seq_len, -1, -1, ...]


    # ========================================================================
    # NEW: Exclude EOS from the valid set (EOS should be predicted exactly, not as part of set)
    # ========================================================================
    # Create mask for EOS tokens in future positions
    # Shape: (B, L, L) - True where future token is EOS
    is_eos_future = (labels_expanded == eos_token_id)

    # Remove EOS from valid_future_mask
    # This makes EOS tokens NOT contribute to the set-loss
    # They will only be predicted when they are the ONLY remaining token
    valid_future_mask = valid_future_mask & ~is_eos_future
    # Explanation:
    # Before: valid_future_mask[0][i] might include EOS among valid predictions
    # After:  valid_future_mask[0][i] excludes EOS from the set
    # Result: When predicting, model won't see EOS as "any valid token"
    #         EOS only gets predicted when it's the last token (standard behavior)


    # Build boolean mask per vocab id using scatter_
    valid_token_mask = torch.zeros((B, L, V), dtype=torch.bool, device=device)
    scatter_idx = labels_expanded.clone()
    scatter_idx[scatter_idx < 0] = 0  # convert -1 to dummy 0, to ignore the positions

    valid_token_mask.scatter_(dim=2, index=scatter_idx, src=valid_future_mask)  # mark valid vocab positions
    # (B, L, V)(16, 21, 1932)
    # valid_token_mask[0][0]
    # [PAD, BOS, EOS, UNK, ---angle---, ---grade---, ---holds---]
    # valid_token_mask[0][0], sum=12 < seq_len
    # [FALSE, TRUE, FALSE, ---one True angle, ---one True grade, ---several True holds---]
    #              ^^^^^ NOTE: EOS is now FALSE (excluded from set)

    # valid_token_mask[0][1], sum=11
    # [FALSE, FALSE, FALSE, ---one True angle, ---one True grade, ---several True holds---]
    #         ^^^^^ BOS excluded    ^^^^^ EOS excluded

    # valid_token_mask[0][2], sum=10
    # [FALSE, FALSE, FALSE, ---all False angle, ---one True grade, ---several True holds---]

    # valid_token_mask[0][3], sum=9
    # [FALSE, FALSE, FALSE, ---all False angle, ---all False grade, ---several True holds---]

    # and then the hold tokens ...

    # note, for next example in batch:
    # valid_token_mask[1][0], sum=17 < seq_len
    # [FALSE, TRUE, FALSE, ---one True angle, ---one True grade, ---several True holds---]

    valid_token_mask[:, :, 0] &= (labels_expanded[:, :, 0] != 0)  # fix dummy zeros if any


    # ========================================================================
    # NEW: Add back EOS prediction when it's the only valid next token
    # ========================================================================
    # Find positions where the next token should be EOS
    # Shape: (B, L)
    next_is_eos = (shift_labels == eos_token_id)

    # At these positions, enable EOS in the vocabulary mask
    # This allows standard cross-entropy loss for EOS prediction
    valid_token_mask[:, :, eos_token_id] |= next_is_eos
    # Explanation:
    # If position i should predict EOS (next_is_eos[b,i] = True):
    #   - Set valid_token_mask[b, i, eos_token_id] = True
    #   - This makes ONLY EOS valid at this position
    #   - Standard single-target prediction for EOS!


    # Filter(/mask) log_probs for only valid positions
    masked_log_probs = torch.where(valid_token_mask, log_probs, torch.full_like(log_probs, -1e9))
    # log_probs[0][0] =>        [-7.6137, -6.4946, -7.7223, ...] the usual
    # valid_token_mask[0][0] => [FALSE,    TRUE,    FALSE,   ...] (BOS and EOS excluded)
    # masked_log_probs[0][0] => [-1e9,    -6.4946,  -1e9,   ...]


    # Combine log(P(token1) + P(token2) + ... + P(tokenN))
    log_sum = torch.logsumexp(masked_log_probs, dim=-1)  # (B, L)
    # For positions with multiple valid tokens: log(P1 + P2 + ... + PN) = set-loss
    # For positions with only EOS valid: log(P_eos) = standard single-target loss


    # log(P1) + log(P2) + ... + log(PN)
    # count_valid = valid_token_mask.sum(dim=-1).clamp(min=1)
    # log_sum = masked_log_probs.sum(dim=-1) / count_valid


    # Only keep positions that had at least one valid target
    has_valid = valid_token_mask.any(dim=-1) # Shape: (B, L)

    # average negative log-likelihood
    loss = -log_sum[has_valid].mean()
    # loss = -log_sum[has_valid].sum() / valid_positions.sum()


    if print_debug:
        print("\n" + "="*70)
        print("DEBUG: Loss Computation Details")
        print("="*70)
        print(f"\nSequence 0 breakdown:")
        for i in range(min(8, L)):
            if has_valid[0, i]:
                valid_tokens = [j for j in range(V) if valid_token_mask[0, i, j]]
                num_valid = len(valid_tokens)
                is_eos_pos = next_is_eos[0, i].item()
                print(f"  Pos {i}: {num_valid} valid tokens | Target={shift_labels[0,i].item():4d} | "
                      f"EOS-only={is_eos_pos} | log_sum={log_sum[0,i].item():7.3f}")


    return loss




In [118]:
class KilterGPT(nn.Module):
    """GPT-2 model for generating Kilter Board climbing routes."""
    def __init__(
        self,
        vocab_size: int,
        n_embd: int = 192,
        n_head: int = 3,
        n_layer: int = 3,
        n_positions: int = 128,
        dropout: float = 0.1
        ):
        super().__init__()
        config = GPT2Config(
            vocab_size=vocab_size,
            n_embd=n_embd,
            n_head=n_head,
            n_layer=n_layer,
            n_positions=n_positions,
            n_ctx=n_positions,
            resid_pdrop=dropout,
            embd_pdrop=dropout,
            attn_pdrop=dropout,
        )
        self.model = GPT2LMHeadModel(config)
        self.config = config
    def forward(self, input_ids, attention_mask=None, labels=None, **kwargs):
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask, labels=None)

        logits = outputs.logits  # (B, L, V) (batch_size, seq_len, vocab_size)

        # shift predictions and labels for next-token prediction
        shift_logits = logits[..., :-1, :].contiguous()   # predict token at t+1 using tokens up to t
        shift_labels = labels[..., 1:].contiguous()       # label for position t is token at t+1

        loss = any_of_next_token_loss_vectorized(shift_logits, shift_labels, ignore_index=-100)
        outputs.loss = loss

        return outputs

In [120]:
model = KilterGPT(
    vocab_size=tokenizer.vocab_size,
    n_embd=256,
    n_head=4,
    n_layer=6,
    n_positions=128,
    dropout=0.1
    )
training_args = TrainingArguments(
    output_dir=OUT_DIR,
    eval_strategy="steps",
    save_strategy="steps",
    save_total_limit=3,
    overwrite_output_dir=True,
    logging_steps=100,  # Log more frequently for wandb
    eval_steps=3000,
    save_steps=3000,
    num_train_epochs=10,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=1,
    learning_rate=1e-5,
    weight_decay=0.01,
    adam_beta1=0.9,
    adam_beta2=0.999,
    report_to="wandb",  # Enable wandb reporting
    remove_unused_columns=False,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_dir=f"{OUT_DIR}/logs",
    load_best_model_at_end=True,
    dataloader_pin_memory=False,
    save_safetensors=False,
    run_name=run_name,
    )
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=datasets["train"],  ### back to train_set without augment
    eval_dataset=datasets["val"],
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)],
    )

In [ ]:
trainer.train()

In [ ]:
test_loss_gen(trainer, model)

# new loss+aug

In [ ]:
augmented_dataset = AugmentedRouteDataset(datasets["train"], tokenizer, num_augments=3)
trainer.train_dataset = augmented_dataset

In [ ]:
trainer.train()

In [ ]:
test_loss_gen(trainer, model)

# draft---debug loss

In [81]:
device = next(model.parameters()).device
# INSPECTIGN THE FIRST BATCH (with size B=16)
batch = next(iter(trainer.get_train_dataloader()))
# print({k: v.shape for k, v in batch.items()})

batch = {k: v.to(device) for k, v in batch.items()}

model.eval()
with torch.no_grad():
    outputs = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        labels=batch["labels"],
    )

logits = outputs.logits.detach().cpu()
labels = batch["labels"].cpu()

# loss_loop = any_of_next_token_loss(logits, labels, print_debug=True)
loss_vec  = any_of_next_token_loss_vectorized(logits, labels, print_debug=True)
loss_vec

tensor(4.2691e+11)

In [82]:
vocab_size = 6
seq_len = 3
batch_size = 1

logits = torch.tensor([
    [
        [1.0, 2.0, 3.0, 0.0, -1.0, 0.5],   # position 0
        [2.0, 1.0, 0.5, -0.5, 3.0, 0.0],   # position 1
        [0.0, 1.0, 0.0, 2.0, -1.0, 3.0]    # position 2
    ]
], dtype=torch.float)

# labels for the same sequence (token ids)
# Shape: (1, 3)
labels = torch.tensor([[1, 3, 5]])

# 🔥 Run the loss with debug prints
loss = any_of_next_token_loss_vectorized(logits, labels, print_debug=False)
loss

tensor(2.6667e+09)

In [83]:

# Get the train dataloader from your Trainer
train_dataloader = trainer.get_train_dataloader()

# Grab one batch
batch = next(iter(train_dataloader))

# Inspect what’s inside
for k, v in batch.items():
    print(k, v.shape)

# Move model to device and set to eval
model = trainer.model
device = next(model.parameters()).device
model.eval()

# Move batch tensors to device
input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)
labels = batch["labels"].to(device)

# Forward pass (disable gradient)
with torch.no_grad():
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels,
    )

# Extract logits
logits = outputs.logits  # shape (B, L, V)
print(f"logits shape: {logits.shape}")

# Also inspect a slice
print("Example logits[0, 0, :10]:", logits[0, 0, :10])
print("Example labels[0, :10]:", labels[0, :10])

# Optionally, compute your custom loss
shift_logits = logits[..., :-1, :].contiguous()
shift_labels = labels[..., 1:].contiguous()

loss = any_of_next_token_loss_vectorized(shift_logits, shift_labels, print_debug=False)
print("Custom loss:", loss.item())


input_ids torch.Size([16, 21])
token_type_ids torch.Size([16, 21])
attention_mask torch.Size([16, 21])
labels torch.Size([16, 21])
logits shape: torch.Size([16, 21, 1932])
Example logits[0, 0, :10]: tensor([ 0.1794,  1.1938, -0.1275, -0.4396,  0.0374,  0.3838,  0.1043, -0.0316,
         0.1946, -0.1256], device='cuda:0')
Example labels[0, :10]: tensor([   1,    6,   21,  593,  746,  938,  950, 1070, 1271, 1484],
       device='cuda:0')
Custom loss: 448079724544.0


In [84]:
import torch.nn.functional as F

idx = 3
input_decoded = tokenizer.decode(input_ids[idx], skip_special_tokens=True)
print(f"\n🧩 Input sequence:\n{input_decoded}\n")

# Compute probabilities for this example
probs = F.softmax(logits[idx], dim=-1)
topk = torch.topk(probs, 3, dim=-1)  # top-3 predictions at each position

print("🔍 Token-level predictions (first 10 tokens):\n")
for i in range(10):  # first 10 tokens
    target_id = labels[idx, i].item()
    target_str = tokenizer.decode([target_id])

    preds_ids = topk.indices[i]
    preds_probs = topk.values[i]
    preds = [(tokenizer.decode([pid.item()]), float(pprob)) for pid, pprob in zip(preds_ids, preds_probs)]

    preds_display = ", ".join([f"{tok} ({p:.5f})" for tok, p in preds])
    print(f"Target: {target_str:10} | Top-3: {preds_display}")



🧩 Input sequence:
angle45 grade18 start1131 start1133 hand1181 hand1183 hand1214 hand1231 hand1263 hand1316 hand1349 finish1386 feet1452 feet1455 feet1493 feet1532

🔍 Token-level predictions (first 10 tokens):

Target: [BOS]      | Top-3: [BOS] (0.00161), start1275 (0.00129), hand1578 (0.00122)
Target: angle45    | Top-3: feet1340 (0.00174), feet1507 (0.00147), feet1264 (0.00133)
Target: grade18    | Top-3: grade18 (0.00136), finish1265 (0.00132), feet1340 (0.00125)
Target: start1131  | Top-3: start1131 (0.00215), feet1284 (0.00145), start1537 (0.00142)
Target: start1133  | Top-3: start1133 (0.00157), start1585 (0.00136), start1254 (0.00127)
Target: hand1181   | Top-3: hand1181 (0.00158), hand1522 (0.00148), start1183 (0.00141)
Target: hand1183   | Top-3: hand1565 (0.00145), feet1350 (0.00144), hand1183 (0.00142)
Target: hand1214   | Top-3: hand1214 (0.00149), finish1087 (0.00130), hand1453 (0.00126)
Target: hand1231   | Top-3: hand1231 (0.00166), finish1106 (0.00143), feet1207 (0.001

# draft---augment

- in dataset, usually, start holds come first
- e.g. sequence that is actually climbed: [BOS, angle40, grade15, start111, start999, hand222, foot444, hand333, finish777, EOS, (PAD)]
- but the model should be able to generate like this:
- given sequence [BOS, angle40, grade15, hand222, foot444]
    -  predict any of [start111, start999, and333, finish777, EOS] since they all complete the climbing route
    - order in the token is invariant(irrelavant) to the climbs after decoding
- therefore, training set should have shuffled sequence of holds as new augmented examples
- we'll skip [BOS, grade, angle, EOS] and shuffle the holds, append to the train set, maybe 2-3x

# draft---training

In [122]:
eval_dataloader = trainer.get_eval_dataloader()

eval_loss_manual = []
device = next(model.parameters()).device

for batch in eval_dataloader:
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
        outputs = model(**batch)
        eval_loss_manual.append(outputs["loss"].item())

manual_eval_loss = sum(eval_loss_manual) / len(eval_loss_manual)
print(f"Manual eval loss: {manual_eval_loss:.9f}")


Manual eval loss: 4.921521980


In [123]:
with torch.no_grad():
    out = model(**batch)
    custom_loss = any_of_next_token_loss_vectorized(out["logits"], batch["labels"])
print("Model loss:", out["loss"].item(), "Manual loss:", custom_loss.item())


Model loss: 4.867768287658691 Manual loss: 3.7657437324523926


In [124]:
import torch

# Get the train dataloader from your Trainer
train_dataloader = trainer.get_train_dataloader()

# Grab one batch
batch = next(iter(train_dataloader))

# Inspect what’s inside
for k, v in batch.items():
    print(k, v.shape)

# Move model to device and set to eval
model = trainer.model
device = next(model.parameters()).device
model.eval()

# Move batch tensors to device
input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)
labels = batch["labels"].to(device)

# Forward pass (disable gradient)
with torch.no_grad():
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels,
    )

# Extract logits
logits = outputs['logits']  # shape (B, L, V)
print(f"logits shape: {logits.shape}")

# Also inspect a slice
print("Example logits[0, 0, :10]:", logits[0, 0, :10])
print("Example labels[0, :10]:", labels[0, :10])

# Optionally, compute your custom loss
shift_logits = logits[..., :-1, :].contiguous()
shift_labels = labels[..., 1:].contiguous()

loss = any_of_next_token_loss_vectorized(shift_logits, shift_labels, print_debug=False)
print("Custom loss:", loss.item())


input_ids torch.Size([16, 21])
token_type_ids torch.Size([16, 21])
attention_mask torch.Size([16, 21])
labels torch.Size([16, 21])
logits shape: torch.Size([16, 21, 1932])
Example logits[0, 0, :10]: tensor([-0.7084, -0.1325,  1.3912, -0.1808,  3.3252,  3.4462,  5.1729,  3.7805,
         6.4167,  5.5452], device='cuda:0')
Example labels[0, :10]: tensor([   1,    6,   21,  593,  746,  938,  950, 1070, 1271, 1484],
       device='cuda:0')
Custom loss: 3.9300506114959717


In [125]:
import torch.nn.functional as F

idx = 3
input_decoded = tokenizer.decode(input_ids[idx], skip_special_tokens=True)
print(f"\n🧩 Input sequence:\n{input_decoded}\n")

# Compute probabilities for this example
probs = F.softmax(logits[idx], dim=-1)
topk = torch.topk(probs, 3, dim=-1)  # top-3 predictions at each position

print("🔍 Token-level predictions (first 10 tokens):\n")
for i in range(10):  # first 10 tokens
    target_id = labels[idx, i].item()
    target_str = tokenizer.decode([target_id])

    preds_ids = topk.indices[i]
    preds_probs = topk.values[i]
    preds = [(tokenizer.decode([pid.item()]), float(pprob)) for pid, pprob in zip(preds_ids, preds_probs)]

    preds_display = ", ".join([f"{tok} ({p:.5f})" for tok, p in preds])
    print(f"Target: {target_str:10} | Top-3: {preds_display}")



🧩 Input sequence:
angle45 grade18 start1131 start1133 hand1181 hand1183 hand1214 hand1231 hand1263 hand1316 hand1349 finish1386 feet1452 feet1455 feet1493 feet1532

🔍 Token-level predictions (first 10 tokens):

Target: [BOS]      | Top-3: angle40 (0.24827), angle45 (0.10385), angle50 (0.07775)
Target: angle45    | Top-3: grade22 (0.07626), grade23 (0.05586), grade20 (0.04966)
Target: grade18    | Top-3: feet1081 (0.01812), start1149 (0.01599), feet1131 (0.01239)
Target: start1131  | Top-3: start1163 (0.02141), start1149 (0.01906), feet1133 (0.01009)
Target: start1133  | Top-3: start1163 (0.02195), start1149 (0.01442), start1169 (0.00736)
Target: hand1181   | Top-3: hand1215 (0.01441), hand1234 (0.01153), hand1236 (0.00969)
Target: hand1183   | Top-3: hand1215 (0.01562), hand1234 (0.01409), hand1236 (0.01288)
Target: hand1214   | Top-3: hand1234 (0.02049), hand1236 (0.01786), hand1215 (0.01425)
Target: hand1231   | Top-3: hand1269 (0.02017), hand1267 (0.01597), hand1234 (0.01578)
Targe

In [126]:
import torch
import torch.nn.functional as F

def visualize_future_mask_behavior(model, tokenizer, dataset, sample_idx=0, top_k=3):
    device = next(model.parameters()).device

    # --- fetch one sample safely ---
    example = dataset[sample_idx]
    input_ids = torch.tensor(example["input_ids"], device=device).unsqueeze(0)

    # if labels not provided, assume equal to input_ids (typical LM setup)
    if "labels" in example:
        labels = torch.tensor(example["labels"], device=device).unsqueeze(0)
    else:
        labels = input_ids.clone()

    # --- forward ---
    with torch.no_grad():
        outputs = model(input_ids=input_ids, labels=labels)
        logits = outputs['logits']

    # --- softmax for probabilities ---
    probs = F.softmax(logits, dim=-1)

    # --- decode ---
    tokens = [tokenizer.decode([t]) for t in input_ids[0]]
    targets = [tokenizer.decode([t]) for t in labels[0]]

    print("\n=== Visualization of causal mask behavior ===")
    print(f"Sample idx: {sample_idx}\n")

    for i in range(len(tokens)):
        context_str = tokenizer.decode(input_ids[0, :i], skip_special_tokens=True)
        next_token = targets[i]
        top_preds = torch.topk(probs[0, i], k=top_k)
        pred_tokens = [tokenizer.decode([tid]) for tid in top_preds.indices]

        print(f"\n--- Step {i} ---")
        print(f"Context (0→{i}): {repr(context_str)}")
        print(f"Target token: {repr(next_token)}")
        print("Top model predictions:")
        for j, (tok, prob) in enumerate(zip(pred_tokens, top_preds.values)):
            print(f"   {j+1:2d}. {repr(tok):15s}  p={prob.item():.4f}")

    print("\n✅ Done — future masking validated (each step only sees prior context).")


# --- ✅ call on your dataset ---
visualize_future_mask_behavior(
    model=trainer.model,
    tokenizer=tokenizer,
    dataset=trainer.train_dataset,
    sample_idx=3
)



=== Visualization of causal mask behavior ===
Sample idx: 3


--- Step 0 ---
Context (0→0): ''
Target token: '[BOS]'
Top model predictions:
    1. 'angle40'        p=0.2483
    2. 'angle45'        p=0.1039
    3. 'angle50'        p=0.0778

--- Step 1 ---
Context (0→1): ''
Target token: 'angle45'
Top model predictions:
    1. 'grade22'        p=0.0763
    2. 'grade23'        p=0.0559
    3. 'grade20'        p=0.0497

--- Step 2 ---
Context (0→2): 'angle45'
Target token: 'grade21'
Top model predictions:
    1. 'feet1081'       p=0.0164
    2. 'start1149'      p=0.0161
    3. 'feet1131'       p=0.0118

--- Step 3 ---
Context (0→3): 'angle45 grade21'
Target token: 'feet1111'
Top model predictions:
    1. 'start1149'      p=0.0194
    2. 'start1163'      p=0.0145
    3. 'feet1133'       p=0.0092

--- Step 4 ---
Context (0→4): 'angle45 grade21 feet1111'
Target token: 'start1143'
Top model predictions:
    1. 'start1163'      p=0.0162
    2. 'hand1215'       p=0.0095
    3. 'hand1198'       

# draft---test

Manual test loss: 7.6337

--- Example 0 ---
Context: [BOS] angle50 grade20 feet1117 start1131 start1163 hand1234 hand1247 hand1302 hand1350 finish1389 [EOS]
Target: [BOS] angle50 grade20 feet1117 start1131 start1163 hand1234 hand1247 hand1302 hand1350 finish1389 [EOS]
Predicted: feet1100 angle50 grade20 feet1117 start1131 hand1158 hand1158 feet1077 start1367 finish1337 finish1315 [EOS] hand1158 [PAD] start1238 hand1158 feet1100 finish1591 [PAD] [PAD] [PAD] [PAD]
Tokens: ['[BOS]', 'angle50', 'grade20', 'feet1117', 'start1131', 'start1163', 'hand1234', 'hand1247', 'hand1302', 'hand1350', 'finish1389', '[EOS]']

--- Example 1 ---
Context: [BOS] angle50 grade24 feet1181 start1184 start1188 hand1254 hand1283 hand1285 hand1314 hand1371 hand1386 finish1392 feet1455 feet1529 feet1580 [EOS]
Target: [BOS] angle50 grade24 feet1181 start1184 start1188 hand1254 hand1283 hand1285 hand1314 hand1371 hand1386 finish1392 feet1455 feet1529 feet1580 [EOS]
Predicted: feet1100 angle50 start1321 feet1100 fee

In [140]:
# assume tokenizer & model are loaded from your fine-tuned checkpoint
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.model
model.to(device)
model.eval()

test_prompts = [
    "[BOS]",
    "[BOS] angle40",
    "[BOS] angle40 grade15",
    "[BOS] angle40 grade15 start1139",
]

# generation config
gen_kwargs = dict(
    max_new_tokens=64,
    do_sample=False,        # deterministic for inspection
    temperature=1.0,
    top_p=0.9,
    pad_token_id=tokenizer.eos_token_id,
    eos_token_id=tokenizer.convert_tokens_to_ids("[EOS]"),
)

for i, prompt in enumerate(test_prompts):
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output = model.generate(input_ids, **gen_kwargs)

    decoded = tokenizer.decode(output[0], skip_special_tokens=False)
    print(f"\n--- Example {i} ---")
    print(f"Context: {prompt}")
    print(f"Generated: {decoded}")

    # Optional: visualize token-by-token
    tokens = tokenizer.convert_ids_to_tokens(output[0])
    print("Tokens:", tokens)


The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



--- Example 0 ---
Context: [BOS]
Generated: [BOS] [BOS] [EOS] [EOS]
Tokens: ['[BOS]', '[BOS]', '[EOS]', '[EOS]']

--- Example 1 ---
Context: [BOS] angle40
Generated: [BOS] [BOS] angle40 [EOS] feet1373 feet1373 feet1373 feet1373 feet1220 feet1220 feet1220 feet1220 feet1220 feet1220 feet1220 feet1220 feet1220 feet1220 feet1220 feet1220 feet1220 feet1220 feet1220 feet1220 feet1220 feet1220 feet1220 feet1220 feet1220 feet1220 feet1220 feet1220 feet1220 feet1220 feet1220 feet1220 feet1220 feet1220 feet1220 feet1220 feet1220 feet1220 feet1220 feet1220 feet1220 feet1220 feet1220 feet1220 feet1220 feet1220 feet1220 feet1220 feet1220 feet1220 feet1220 feet1220 feet1220 feet1220 feet1220 feet1220 feet1220 feet1220 feet1220 feet1220 feet1220 feet1220 feet1220 feet1220
Tokens: ['[BOS]', '[BOS]', 'angle40', '[EOS]', 'feet1373', 'feet1373', 'feet1373', 'feet1373', 'feet1220', 'feet1220', 'feet1220', 'feet1220', 'feet1220', 'feet1220', 'feet1220', 'feet1220', 'feet1220', 'feet1220', 'feet1220', 'fee